# Text cleaning, hashtag normalisation, English translation and a CatBoost text model

In [ ]:
DATA_DIR = "../data"

In [ ]:
import pandas as pd
import numpy as np
import re

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback
)


from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    # text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE [^\]]*\]', '', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9 ]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")

    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")

train_df['text'] = train_df['text'].apply(remove_noise)
train_df['text'] = train_df['text'].apply(normalize_word)
train_df = drop_duplicated(train_df)
train_df.head()

Df size: (4583, 3)
After dropping empty text rows, df size:  (4574, 3)
After dropping literal duplicated rows, df size:  (4297, 3)
After dropping text duplicated rows, df size:  (4252, 3)


,text,label,label_cnt
0,Jangan Golput pilih Capres No 3 Ganjar Pranowo...,Geografi,20
2,Itu juga pulau yang jalan krna emg udah jadi d...,Geografi,20
3,Jadi mau pindah nanti ke IKN kalo dapet rezeki...,Geografi,20
4,kampung ku di aeange deket bukit tinggi kamoun...,Geografi,20
5,Umat Islam rata rata mendukung Anies Baswedan ...,Geografi,20


In [ ]:
def translate_hashtags_in_dataframe(df, column_name):
    translation_map = {
        "#JNK": "#JNK",
        "#GanjarMahfudRebound": "#GanjarMahfudRebound",
        "#GanjarMahfud2024": "#GanjarMahfud2024",
        "#Coblos3": "#Vote3",
        "#02Melanjutkan": "#Continue02",
        "#AMINAjaDulu": "#JustAMINAFirst",
        "#L3bihbaik": "#Better3",
        "#3GM": "#3GM",
        "#IndonesiaSentris": "#IndonesiaCentric",
        "#IndonesiaHijau": "#GreenIndonesia",
        "#Ganjar": "#Ganjar",
        "#MahfudLebihBaik3": "#MahfudBetter3",
        "#DebatCapres": "#PresidentialDebate",
        "#AniesMuhaimin2024": "#AniesMuhaimin2024",
        "#TempoNasional": "#TempoNational",
        "#TabrakProf": "#HitTheProf",
        "#AsalBukanPrabowo": "#AnyoneButPrabowo",
        "#JabarTetapUntuk02": "#WestJavaStillFor02",
        "#SahabatGanjar": "#GanjarFriends",
        "#RembuGanjar": "#GanjarMeeting",
        "#02UntukMelanjutkan": "#02ToContinue",
        "#GanjarPresiden": "#GanjarPresident",
        "#DesakAnies": "#UrgeAnies",
        "#GanjarUntukSemua": "#GanjarForAll",
        "#GanjarMahfudM3nang": "#GanjarMahfudWin3",
        "#3MandatGanjarMahfud": "#3MandatesGanjarMahfud",
        "#PrabowoGemoy": "#CutePrabowo",
        "#JabarTetapPakBowo": "#WestJavaStillForBowo",
        "#PDIPerjuangan": "#PDIPerjuangan",
        "#Pilpres2024": "#PresidentialElection2024",
        "#MenataMasaDepan": "#ShapingTheFuture",
        "#news": "#news",
        "#GanjarPranowo": "#GanjarPranowo",
        "#MEnangkanGAnjar": "#WinGanjar",
        "#MenangSeputaran": "#WinningAround",
        "#update": "#update",
        "#kodekita08": "#codeour08",
        "#TemanAlamGanjar": "#GanjarNatureFriends",
        "#text": "#text",
        "#SatSetLebihBaik": "#BetterQuick",
        "#KratonBersamaRakyat": "#PalaceWithPeople",
        "#Pemilu2024": "#Election2024",
        "#nazarpemilu": "#ElectionVow",
        "#HardNews": "#HardNews",
        "#CariBeritaditvOne": "#FindNewsAtTvOne",
        "#AniesBaswedan": "#AniesBaswedan",
        "#Politik": "#Politics",
        "#tvOneTVPemilu": "#tvOneElectionTV",
        "#MauMemilih": "#WantToVote",
        "#Nomor3Pilihanku": "#MyChoiceIsNumber3",
        "#M3nangDebat4": "#WinDebate4",
        "#SatSet": "#Quick",
        "#PastiLebihBaik": "#DefinitelyBetter",
        "#aniesbaswedan": "#aniesbaswedan",
        "#newsupdate": "#newsupdate",
        "#prabowo": "#prabowo",
        "#GanjarSatSet": "#GanjarQuick",
        "#MahfudTasTes": "#MahfudTasTest",
        "#AniesPresiden2024": "#AniesPresident2024",
        "#GanjarMahfud": "#GanjarMahfud",
        "#JelasSama02": "#ClearWith02",
        "#Capres": "#PresidentialCandidate",
        "#pemilu2024": "#election2024",
        "#AtikohIbuNegara": "#AtikohFirstLady",
        "#NewsOne": "#NewsOne",
        "#Cawapres": "#VicePresidentialCandidate",
        "#Mahfud": "#Mahfud",
        "#HaveAniesDay": "#HaveAniesDay",
        "#TempoBisnis": "#TempoBusiness",
        "#PresidenRakyat": "#People'sPresident",
        "#DialogGanjar": "#GanjarDialogue",
        "#Prabowo": "#Prabowo",
        "#arsjadrasjid": "#arsjadrasjid",
        "#PersatuanNasional": "#NationalUnity",
    }

    def translate_text(text):
        words = text.split()
        translated_words = [translation_map.get(word, word) for word in words]
        return ' '.join(translated_words)

    df[column_name] = df[column_name].apply(translate_text)
    return df

train_df = translate_hashtags_in_dataframe(train_df, 'text')

In [ ]:
# reset index of df
train_df.reset_index(drop=True, inplace=True)

In [ ]:
from deep_translator import GoogleTranslator

def translate_to_en(text):
    return GoogleTranslator(source='auto', target='en').translate(text)

train_df['clean_text_en'] = train_df['text'].apply(translate_to_en)

In [ ]:
train_df.to_csv(f"{DATA_DIR}/train_cleaned_en_akmal.csv", index=False)

In [ ]:
train_df

,text,label,label_cnt
0,Jangan Golput pilih Capres No 3 Ganjar Pranowo...,Geografi,20
1,Itu juga pulau yang jalan krna emg udah jadi d...,Geografi,20
2,Jadi mau pindah nanti ke IKN kalo dapet rezeki...,Geografi,20
3,kampung ku di aeange deket bukit tinggi kamoun...,Geografi,20
4,Umat Islam rata rata mendukung Anies Baswedan ...,Geografi,20
...,...,...,...
4247,Anies apresiasi jasa pendidik dengan bebaskan ...,Politik,2972
4248,Anies Bansos Diberikan Ikut Kalender Kebutuhan...,Politik,2972
4249,Anies nothing to lose harus total war targetny...,Politik,2972
4250,Closing Statement pak Anies di acara KADIN sem...,Politik,2972


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text
import umap.umap_ as umap



# Define custom stopwords
stop_words = list(text.ENGLISH_STOP_WORDS)

# Create TF-IDF Vectorizer with stopword removal
vectorizer = TfidfVectorizer(max_features=1000, stop_words=stop_words)
tfidf_matrix = vectorizer.fit_transform(train_df['text'])

# Convert the TF-IDF matrix to a DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# Perform UMAP dimensionality reduction to 50 dimensions
reducer = umap.UMAP(n_components=50, random_state=42)
umap_features = reducer.fit_transform(tfidf_df)

# Convert the reduced features to a DataFrame for better visualization
umap_df = pd.DataFrame(umap_features, columns=[f"UMAP_{i}" for i in range(50)])

# Add the UMAP features to the original DataFrame
for i in range(50):
    train_df[f'UMAP_{i}'] = umap_df[f'UMAP_{i}']

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [ ]:
train_df

                                                   text     label  label_cnt  \
0     Jangan Golput pilih Capres No 3 Ganjar Pranowo...  Geografi         20   
1     Itu juga pulau yang jalan krna emg udah jadi d...  Geografi         20   
2     Jadi mau pindah nanti ke IKN kalo dapet rezeki...  Geografi         20   
3     kampung ku di aeange deket bukit tinggi kamoun...  Geografi         20   
4     Umat Islam rata rata mendukung Anies Baswedan ...  Geografi         20   
...                                                 ...       ...        ...   
4247  Anies apresiasi jasa pendidik dengan bebaskan ...   Politik       2972   
4248  Anies Bansos Diberikan Ikut Kalender Kebutuhan...   Politik       2972   
4249  Anies nothing to lose harus total war targetny...   Politik       2972   
4250  Closing Statement pak Anies di acara KADIN sem...   Politik       2972   
4251  Capres nomor urut 3 Ganjar Pranowo menanggapi ...   Politik       2972   

         UMAP_0     UMAP_1    UMAP_2   

In [ ]:
# create feature using tf-idf 1000 column

In [ ]:
# drop label cnt
train_df.drop(['label_cnt'], axis=1, inplace=True)

In [ ]:
train_df.tail(30)

                                                   text    label     UMAP_0  \
4222  Terlalu Banyak Lihat Catatan Saat Debat Cak Im...  Politik   9.907032   
4223  prabowo Rakyat Berebut Bingkisan Anies Rakyat ...  Politik   9.787403   
4224  PRESIDEN JOKOWI GA TAHU MALU GA PUNYA FATSUM P...  Politik   9.822333   
4225  Bro kalo lo mau pilihan yang beda buat Pilpres...  Politik   9.597383   
4226  Nelayan Indonesia nggak bisa terus terjebak da...  Politik  10.049279   
4227  Dalam visinya Ganjar Pranowo dan Mahfud berupa...  Politik   9.714923   
4228  Anies Baswedan berkunjung ke kediaman raja dan...  Politik   9.977116   
4229  Pastikan kamu sekeluarga datang ke Tempat Pemu...  Politik   9.947724   
4230  Capres Cawapres Ganjar Pranowo Mahfud Mahfud s...  Politik   9.451355   
4231  Keberhasilan Pertahanan dan kemakmuran ekonomi...  Politik   9.799691   
4232  Joget joget memang bisa menyenangkan tapi pemu...  Politik   9.683279   
4233  Siap mendukung Pak Prabowo biar tidak ada koru

In [ ]:
# Label distribution
label_counts = {
    'Politik': 2858,
    'Sosial Budaya': 391,
    'Ekonomi': 267,
    'Pertahanan dan Keamanan': 263,
    'Ideologi': 258,
    'Sumber Daya Alam': 140,
    'Demografi': 56,
    'Geografi': 19
}

# Calculate weights (inverse of frequency)
total_samples = sum(label_counts.values())
weights = {label: total_samples / count for label, count in label_counts.items()}

# Normalize weights
max_weight = max(weights.values())
weights = {label: weight / max_weight for label, weight in weights.items()}

# Create a mapping of labels to weights
label_mapping = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}

class_weights = [weights[label] for label in label_mapping.keys()]

# mape label
train_df['label'] = train_df['label'].map(label_mapping)

In [ ]:
model_1 = CatBoostClassifier(
    text_features=['text'],
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=0,
    random_state=42,
    class_weights=class_weights,
)


cv_method = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=42)
cv_results_model_1 = cross_val_score(estimator=model_1,  X=train_df.drop(['label', 'label_cnt'], axis=1), y=train_df["label"], cv=cv_method,
                                     verbose=100, scoring='balanced_accuracy')

print("Model 1 - Balanced Accuracy scores: ")
print(cv_results_model_1)
print("Mean: ")
print(cv_results_model_1.mean().round(3))

[CV] START .....................................................................
[CV] END ................................ score: (test=0.462) total time=  10.6s
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:   10.6s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.507) total time=   9.4s
[Parallel(n_jobs=1)]: Done   2 tasks      | elapsed:   20.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.464) total time=   9.5s
[Parallel(n_jobs=1)]: Done   3 tasks      | elapsed:   29.6s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.476) total time=   9.7s
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:   39.3s
[CV] START .....................................................................
[CV] END ..................

In [ ]:
# make all text lowercase
train_df['text'] = train_df['text'].str.lower()

In [ ]:
train_df.to_csv(f'{DATA_DIR}/train_indo_akmal.csv', index=False)